# House Prices – Advanced Regression Analysis
**Dataset:** Kaggle House Prices: Advanced Regression Techniques (`train.csv`)

**Objective:** Predict the final sale price (`SalePrice`) of residential homes in Ames, Iowa, using various features describing the property. We will explore the data, preprocess it, and compare a **Linear Regression** model against a **Decision Tree Regressor**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('train.csv')
df.head()

# Attribute Information
***Key input features used in this analysis:***

1. **OverallQual**: Overall material and finish quality (1–10)
2. **GrLivArea**: Above-ground living area in square feet
3. **GarageCars**: Size of garage in car capacity
4. **GarageArea**: Size of garage in square feet
5. **TotalBsmtSF**: Total square feet of basement area
6. **1stFlrSF**: First floor square feet
7. **FullBath**: Number of full bathrooms above grade
8. **TotRmsAbvGrd**: Total rooms above grade (excluding bathrooms)
9. **YearBuilt**: Original construction year
10. **YearRemodAdd**: Remodel year (same as YearBuilt if no remodel)
11. **LotArea**: Lot size in square feet
12. **MSSubClass**: Type of dwelling involved in the sale

***Target variable:***
- **SalePrice**: The property's sale price in USD (what we are predicting)

In [ ]:
# Basic shape and data types
print('Dataset shape:', df.shape)
print('\nData types:')
print(df.dtypes.value_counts())

In [ ]:
# Statistical summary of numeric features
df.describe()

## Missing Value Analysis
Before modeling, we need to understand which columns have missing data. Columns with very high missing rates (e.g., `PoolQC`, `MiscFeature`, `Alley`) represent rare features and will be dropped. Numeric columns with moderate missing values will be filled with the column median.

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values:')
print(missing)

# Visualise top 15 missing columns
plt.figure(figsize=(12, 5))
missing.head(15).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 15 Columns with Missing Values')
plt.ylabel('Number of Missing Entries')
plt.xlabel('Column')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Data Preprocessing
We select only the **numeric features** for this regression analysis, drop columns with excessive missing values, and impute the remaining NaNs with the column median. This gives us a clean numeric matrix ready for modelling.

In [ ]:
# Select numeric columns only
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Drop columns with >50% missing values
threshold = len(numeric_df) * 0.5
numeric_df = numeric_df.dropna(axis=1, thresh=threshold)

# Fill remaining NaNs with median
numeric_df = numeric_df.fillna(numeric_df.median())

# Drop Id column (not a feature)
numeric_df = numeric_df.drop(columns=['Id'], errors='ignore')

print('Cleaned dataset shape:', numeric_df.shape)
print('Any remaining nulls:', numeric_df.isnull().sum().sum())

## Correlation Heatmap
The heatmap below shows pairwise correlations between all numeric features and the target `SalePrice`. Features with high positive correlation (closer to +1) are stronger predictors of house price. Notable correlations include `OverallQual`, `GrLivArea`, and `GarageCars`.

In [ ]:
# Correlation heatmap
plt.figure(figsize=(18, 14))
corr = numeric_df.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of All Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 features most correlated with SalePrice
top_corr = corr['SalePrice'].abs().sort_values(ascending=False)[1:11]
print('Top 10 features correlated with SalePrice:')
print(top_corr)

plt.figure(figsize=(10, 5))
top_corr.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Top 10 Features Correlated with SalePrice')
plt.ylabel('Absolute Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Distribution of Key Features
Histograms give us a sense of the spread and skewness of our features. `SalePrice` itself is right-skewed, which is typical for housing data — most homes sell at moderate prices, with a long tail of expensive properties.

In [ ]:
# Histogram of SalePrice
plt.figure(figsize=(8, 4))
plt.hist(numeric_df['SalePrice'], bins=50, color='steelblue', edgecolor='black')
plt.title('Distribution of SalePrice')
plt.xlabel('SalePrice (USD)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Histograms of top numeric features
key_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'YearBuilt', 'LotArea']

numeric_df[key_features].hist(bins=30, figsize=(14, 8), color='steelblue', edgecolor='black')
plt.suptitle('Histograms of Key Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Scatter Plots: Top Features vs SalePrice
Scatter plots help us visualise linear (or non-linear) relationships between individual features and `SalePrice`. `GrLivArea` and `OverallQual` show especially strong positive trends.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    axes[i].scatter(numeric_df[feat], numeric_df['SalePrice'], alpha=0.3, color='steelblue', s=10)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('SalePrice')
    axes[i].set_title(f'{feat} vs SalePrice')

plt.suptitle('Feature vs SalePrice Scatter Plots', fontsize=14)
plt.tight_layout()
plt.show()

## Train / Test Split & Feature Scaling
We split the data into **80% training** and **20% testing** sets. Features are then standardised using `StandardScaler` so that no single feature dominates due to its scale (e.g., `LotArea` in sq ft vs `FullBath` count).

In [ ]:
# Define features and target
X = numeric_df.drop(columns=['SalePrice'])
y = numeric_df['SalePrice']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training set size:', X_train.shape)
print('Testing set size:', X_test.shape)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\nFeature scaling applied (StandardScaler)')

## Model 1: Linear Regression
Linear Regression fits a straight-line relationship between each feature and the target. It is interpretable and fast, but assumes a linear relationship which may not fully capture the complexity of housing price data.

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print('=== Linear Regression ===')
print(f'MSE:  {mse_lr:,.2f}')
print(f'RMSE: {rmse_lr:,.2f}')
print(f'R²:   {r2_lr:.4f}')

In [ ]:
# Plot Actual vs Predicted - Linear Regression
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.4, color='steelblue', s=15)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual SalePrice')
plt.ylabel('Predicted SalePrice')
plt.title('Linear Regression: Actual vs Predicted SalePrice')
plt.legend()
plt.tight_layout()
plt.show()

## Model 2: Decision Tree Regressor
Decision Trees partition the feature space into regions and predict the mean SalePrice within each region. They can capture non-linear relationships and feature interactions, but are prone to overfitting without constraints.

In [ ]:
# Train Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train_scaled, y_train)

In [ ]:
# Predict
y_pred_dt = dt_model.predict(X_test_scaled)

# Evaluate
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print('=== Decision Tree Regressor ===')
print(f'MSE:  {mse_dt:,.2f}')
print(f'RMSE: {rmse_dt:,.2f}')
print(f'R²:   {r2_dt:.4f}')

In [ ]:
# Plot Actual vs Predicted - Decision Tree
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_dt, alpha=0.4, color='coral', s=15)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'b--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual SalePrice')
plt.ylabel('Predicted SalePrice')
plt.title('Decision Tree: Actual vs Predicted SalePrice')
plt.legend()
plt.tight_layout()
plt.show()

## Model Comparison
We now compare both models side-by-side using RMSE and R² score. A lower RMSE and a higher R² indicate a better model.

In [ ]:
# Side-by-side comparison
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree'],
    'MSE': [mse_lr, mse_dt],
    'RMSE': [rmse_lr, rmse_dt],
    'R² Score': [r2_lr, r2_dt]
})
print(results.to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Linear Regression', 'Decision Tree'], [rmse_lr, rmse_dt],
            color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('RMSE Comparison (lower is better)')
axes[0].set_ylabel('RMSE')

axes[1].bar(['Linear Regression', 'Decision Tree'], [r2_lr, r2_dt],
            color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_title('R² Score Comparison (higher is better)')
axes[1].set_ylabel('R² Score')
axes[1].set_ylim(0, 1)

plt.suptitle('Model Performance Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Conclusion

In this analysis we explored the **Ames Housing dataset** to predict residential `SalePrice` using machine learning regression models.

**Key findings:**
- The most important numeric predictors of sale price are **OverallQual**, **GrLivArea**, **GarageCars**, **GarageArea**, and **TotalBsmtSF**.
- `SalePrice` is right-skewed — a log transformation could improve model performance in future work.
- **Linear Regression** provides a reasonable baseline but may underfit due to non-linear relationships in the data.
- **Decision Tree Regressor** can capture non-linear patterns but is susceptible to overfitting on the training set.

**Possible improvements:**
- Encode categorical features (e.g., `Neighborhood`, `KitchenQual`) to include more predictors.
- Apply log transformation to `SalePrice` and skewed features.
- Try ensemble models such as **Random Forest** or **Gradient Boosting (XGBoost)** for better performance.